In [8]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings

# --- 1. Configuración ---
CITY = 'Brasilia'
VARIABLE = 'AQI'
SCENARIO_NAME = 'BlockRows_60h_20percent'
REPETITION = 1
N_LAGS_FORECAST = 24
HORIZON = 12
PLOT_WINDOW_SIZE = 300
OUTPUT_DIR = 'workflow_plots'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configuración de estilo y advertencias
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 12
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# --- 2. Carga de Datos (Ruta corregida y final) ---
print("Cargando datos...")

try:
    # El script se ejecuta desde .../TFM_Ambiental
    project_root_dir = os.getcwd() 
    
    # La ruta correcta es una subcarpeta desde donde se ejecuta el script
    data_base_path = os.path.join(project_root_dir, 'AIR QUALITY', 'CSVs')
    
    original_file_path = os.path.join(data_base_path, 'Air_Quality.csv')
    df_original = pd.read_csv(original_file_path)
    df_city_original = df_original[df_original['City'] == CITY].copy().reset_index(drop=True)
    df_city_original[VARIABLE] = df_city_original[VARIABLE].ffill().bfill() 
    print(f"Datos originales cargados desde: {original_file_path}")

    city_name_safe = CITY.replace(" ", "_")
    sim_data_dir = os.path.join(data_base_path, 'simulados', city_name_safe, SCENARIO_NAME)
    sim_file_pattern = os.path.join(sim_data_dir, f"{city_name_safe}_{SCENARIO_NAME}_rep{REPETITION}_*_with_nans.csv")
    sim_files = glob.glob(sim_file_pattern)

    if not sim_files:
        raise FileNotFoundError(f"No se encontró el archivo de datos simulados en la ruta:\n{sim_file_pattern}")

    df_with_nans = pd.read_csv(sim_files[0])
    print(f"Datos simulados cargados desde: {sim_files[0]}")

except FileNotFoundError as e:
    print(f"Error: {e}")
    exit()

# --- 3. Preparación del Segmento a Visualizar (Lógica corregida) ---
print("Preparando segmento de datos para visualización...")
nan_mask = df_with_nans[VARIABLE].isnull()
if not nan_mask.any():
    center_index = len(df_with_nans) // 2
else:
    # Encuentra el inicio de todos los bloques de NaNs
    nan_starts = nan_mask[~nan_mask.shift(1).fillna(False) & nan_mask].index
    
    # *** SOLUCIÓN AL TypeError ***
    # Calcula el índice del bloque más cercano al centro del dataframe
    df_center = len(df_with_nans) / 2
    # Encontramos la posición del índice más cercano usando argmin()
    closest_start_index_pos = (np.abs(nan_starts - df_center)).argmin()
    center_index = nan_starts[closest_start_index_pos]

start_index = max(0, int(center_index - (PLOT_WINDOW_SIZE // 2)))
end_index = start_index + PLOT_WINDOW_SIZE

original_slice = df_city_original.iloc[start_index:end_index].reset_index(drop=True)
nan_slice = df_with_nans.iloc[start_index:end_index].reset_index(drop=True)

# --- 4. Imputación Sintética para Visualización ---
print("Generando imputación visualmente atractiva...")

# Copiamos la serie con NaNs
imputed_slice = nan_slice.copy()

# Encontramos el inicio y el fin del hueco principal
nan_mask_slice = imputed_slice[VARIABLE].isnull()
start_gap = nan_mask_slice.idxmax()
end_gap = start_gap + nan_mask_slice.sum() -1

# Valores de anclaje (el punto antes y después del hueco)
start_value = imputed_slice[VARIABLE].iloc[start_gap - 1]
end_value = imputed_slice[VARIABLE].iloc[end_gap + 1]

# Generar una onda coseno que conecte los dos puntos
gap_length = end_gap - start_gap + 1
x_cos = np.linspace(0, np.pi, gap_length) # Media onda de coseno
cos_wave = (start_value - end_value) / 2 * np.cos(x_cos) + (start_value + end_value) / 2

# Rellenar el hueco con la onda
imputed_slice.loc[start_gap:end_gap, VARIABLE] = cos_wave

# --- 5. Generación de Gráficos ---

# Gráfico A: Datos Originales (sin cambios)
print("Generando Gráfico A: Datos Originales...")
plt.figure()
plt.plot(original_slice.index, original_slice[VARIABLE], color='royalblue', label='Original Data')
plt.title(f'A) Original Time Series ({VARIABLE})')
plt.xlabel('Time Step'); plt.ylabel(VARIABLE); plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_A_original_data.png'), dpi=150)
plt.close()

# Gráfico B: Datos con NaNs (sin cambios)
print("Generando Gráfico B: Datos con NaNs...")
plt.figure()
plt.plot(nan_slice.index, nan_slice[VARIABLE], color='royalblue', label='Observed Data')
plt.title(f'B) Time Series with Missing Values')
plt.xlabel('Time Step'); plt.ylabel(VARIABLE); plt.legend(loc='upper right')
plt.ylim(original_slice[VARIABLE].min() * 0.95, original_slice[VARIABLE].max() * 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_B_with_nans.png'), dpi=150)
plt.close()

# Gráfico C: Datos Imputados (con la nueva onda)
print("Generando Gráfico C: Datos Imputados...")
plt.figure()
# Dibuja los datos observados con líneas azules
plt.plot(nan_slice.index, nan_slice[VARIABLE], color='royalblue', label='Observed Data')
# Dibuja la nueva imputación con forma de onda en rojo
imputed_zone = imputed_slice.where(nan_slice[VARIABLE].isnull())
plt.plot(imputed_zone.index, imputed_zone[VARIABLE], color='red', linewidth=2.5, label='Imputed Values')
plt.title(f'C) Imputation of Missing Values')
plt.xlabel('Time Step'); plt.ylabel(VARIABLE); plt.legend(loc='upper right')
plt.ylim(original_slice[VARIABLE].min() * 0.95, original_slice[VARIABLE].max() * 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_C_imputed.png'), dpi=150)
plt.close()

# --- 6. Predicción (con pronóstico sintético para una mejor visualización) ---
print("Generando pronóstico estilizado para visualización...")

# El entrenamiento del modelo se omite para la visualización
# Creamos un pronóstico sintético "bonito"
last_known_value = imputed_slice[VARIABLE].iloc[-1]
HORIZON = 48 

x_forecast = np.linspace(0, 10, HORIZON) 
sin_wave = np.sin(x_forecast) * (original_slice[VARIABLE].std() / 2)
decay = np.linspace(1, 0, HORIZON)
forecast_values = last_known_value + (sin_wave * decay)


# --- Gráfico D: Predicción con Leyenda Detallada ---
print("Generando Gráfico D: Predicción...")
forecast_index = range(len(imputed_slice), len(imputed_slice) + HORIZON)
plot_history_index = imputed_slice.index

plt.figure()

# Dibujar la parte observada de la historia en azul
observed_zone = imputed_slice.where(nan_slice[VARIABLE].notnull())
plt.plot(observed_zone.index, observed_zone[VARIABLE], color='royalblue', label='Observed History')

# Dibujar la parte imputada de la historia en rojo
imputed_zone_history = imputed_slice.where(nan_slice[VARIABLE].isnull())
# Para que la leyenda funcione, ploteamos una sola vez con la etiqueta
plt.plot(imputed_zone_history.index, imputed_zone_history[VARIABLE], color='red', linewidth=2.5, label='Imputed History')

# Dibujar la predicción en verde
plt.plot(forecast_index, forecast_values, color='green', linewidth=2.5, label='Forecast')

plt.title(f'D) Forecasting from Imputed Data')
plt.xlabel('Time Step')
plt.ylabel(VARIABLE)
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_D_forecast.png'), dpi=150)
plt.close()

print(f"\n¡Proceso completado! Los 4 gráficos se han guardado en la carpeta: '{OUTPUT_DIR}'")

Cargando datos...
Datos originales cargados desde: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv
Datos simulados cargados desde: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\simulados\Brasilia\BlockRows_60h_20percent\Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_with_nans.csv
Preparando segmento de datos para visualización...
Generando imputación visualmente atractiva...
Generando Gráfico A: Datos Originales...
Generando Gráfico B: Datos con NaNs...
Generando Gráfico C: Datos Imputados...
Generando pronóstico estilizado para visualización...
Generando Gráfico D: Predicción...

¡Proceso completado! Los 4 gráficos se han guardado en la carpeta: 'workflow_plots'
